# PID 제어 · 실습


## 학습 목표

<!-- exercise-guidance:control-2/pid-control/objective -->
빠른 응답만 보고 Gain을 고르면 Overshoot나 모델 변화에 취약할 수 있습니다. 서로 다른 성능 지표를 비교합니다.
<!-- /exercise-guidance:control-2/pid-control/objective -->


## P·PI·PID 기준 비교

<!-- exercise-guidance:control-2/pid-control/activity-1 -->
### 시작 조건

Motor는 J=.01,b=.1,K=.01,R=1,L=.5. P=(500,0,0), PI=(500,300,0), PID=(500,300,30)입니다.

### 진행 방법

1. 음의 단위 Feedback으로 세 Closed-loop 모델을 만드세요.
2. t=0~2 s의 Unit Step을 같은 그림에 표시하고 Pole을 먼저 확인하세요.
3. 안정한 모델에 대해 Overshoot·Settling Time·정상상태 오차를 기록하세요.

### 남길 결과

- 제어기 3행의 표와 응답 그림, 역할 비교 짧은 설명.
<!-- /exercise-guidance:control-2/pid-control/activity-1 -->


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import control as ctl
s = ctl.tf('s')
G=.01/((.01*s+.1)*(.5*s+1)+.01**2)
configs={'P':(500,0,0),'PI':(500,300,0),'PID':(500,300,30)}
for name,(kp,ki,kd) in configs.items():
    C=kp+ki/s+kd*s
    T=ctl.feedback(C*G,1)
    print(name,ctl.poles(T))
# TODO: 응답과 성능 지표를 기록하세요.


## 결과 기록·자가 점검

<!-- exercise-guidance:control-2/pid-control/activity-1-record -->
**제어기:**

**Kp:**

**Ki:**

**Kd:**

**안정 여부:**

**Overshoot [%]:**

**Settling Time [s]:**

**정상상태 오차:**

### 확인할 점

- 제어기 이외의 Plant·입력·시간축은 같은가?
- 불안정하거나 수렴이 끝나지 않은 경우 지표 대신 사유를 적었는가?
<!-- /exercise-guidance:control-2/pid-control/activity-1-record -->


## 한 항씩 변화시키기

<!-- exercise-guidance:control-2/pid-control/activity-2 -->
세 경우의 Unit Step과 지표를 같은 방식으로 기록하세요.

### 시작 조건

PID 기준 (500,300,30)에서 Kp만 250,500,750으로 바꿉니다. Ki·Kd는 고정합니다.

### 진행 방법

1. PID 기준에서 Kp=250과 750만 비교하고, Ki=300·Kd=30은 유지하세요.
2. 응답이 빨라진 정도와 Overshoot 변화를 같은 그림·표로 확인하세요.
3. 두 설정 중 하나를 선택하고 선택한 성능 기준을 한 문장으로 적으세요.

### 남길 결과

- 두 설정의 응답 그림과 지표 표. 선택 이유 한 문장.
<!-- /exercise-guidance:control-2/pid-control/activity-2 -->


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import control as ctl
s = ctl.tf('s')
G=.01/((.01*s+.1)*(.5*s+1)+.01**2)
for kp in [250,500,750]:
    T=ctl.feedback((kp+300/s+30*s)*G,1)
    print(kp,ctl.dcgain(T))
# TODO: 시간응답과 두 번째 Gain 비교를 완성하세요.


## 결과 기록·자가 점검

<!-- exercise-guidance:control-2/pid-control/activity-2-record -->
**변경 항:**

**값:**

**Overshoot [%]:**

**Settling Time [s]:**

**정상상태 오차:**

### 확인할 점

- 한 번에 한 Gain만 바꾸었는가?
- 빠르다는 평가를 구체적인 지표로 설명했는가?
<!-- /exercise-guidance:control-2/pid-control/activity-2-record -->


## 입력 한계와 적분 누적

<!-- exercise-guidance:control-2/pid-control/activity-3 -->
### 시작 조건

1차 Plant y′=−y+u, 목표 2→.5(5초에 변경), u∈[0,1], dt=.01 s입니다. Kp=2, Ki=1입니다.

### 진행 방법

1. 제공된 PI 시뮬레이션에서 실제 입력과 제한 전 입력을 함께 그리세요.
2. conditional integration을 켠 경우와 끈 경우를 비교하세요.
3. 목표를 낮춘 뒤 오차가 오래 남는 이유를 적분 상태와 연결해 설명하세요.

### 남길 결과

- 출력·입력·적분 상태의 세 그래프와 비교 표.
<!-- /exercise-guidance:control-2/pid-control/activity-3 -->


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
def simulate(antiwindup):
    t=np.arange(0,12,.01);y=0.;integral=0.;rows=[]
    for now in t:
        ref=2. if now<5 else .5;e=ref-y;raw=2*e+integral;u=np.clip(raw,0,1)
        if not antiwindup or (0<raw<1) or (raw>=1 and e<0) or (raw<=0 and e>0):integral+=e*.01
        y+=(-y+u)*.01;rows.append((ref,y,raw,u,integral))
    return t,np.array(rows)
for flag in [False,True]:
    time,trace=simulate(flag);plt.plot(time,trace[:,1],label=str(flag))
plt.xlabel('Time [s]');plt.ylabel('Output');plt.legend()
# TODO: 입력·적분 상태와 5초 이후의 지표를 추가하세요.


## 결과 기록·자가 점검

<!-- exercise-guidance:control-2/pid-control/activity-3-record -->
**적분 처리:**

**목표 변경 후 IAE:**

**최대 적분값:**

**출력 한계 지속 시간:**

### 확인할 점

- 입력 제한을 적용하기 전후 값을 구분했는가?
- PI Gain까지 바꾸지 않고 적분 처리 방식만 비교했는가?
<!-- /exercise-guidance:control-2/pid-control/activity-3-record -->
